In [31]:
import pandas as pd
import numpy as np
import statsmodels.api as sm_api
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle
import re

In [32]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)

# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'province_id': 'province_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares'
}, inplace=True)

# calculate number of products in each market x nest
df['product_set_size'] = df.groupby(['market_ids', 'nesting_ids'])['product_ids'].transform('count')



In [33]:
################ Prepare policy data net prices of policies
# Import the province policy data
province_policy = pd.read_excel(r'C:\Users\Lenovo\Desktop\Dissertaion\China Data\Demand\Policy\Province_policy.xlsx')

# Import national policy data
national_policy = pd.read_excel(r'C:\Users\Lenovo\Desktop\Dissertaion\China Data\Demand\Policy\National_policy.xlsx')

# Sort province data
def expand_year_range(row):
    years = []
    if pd.notnull(row['year']):
        parts = str(row['year']).split('-')
        if len(parts) == 2:
            start, end = int(parts[0]), int(parts[1])
            years = list(range(start, end + 1))
        else:
            years = [int(parts[0])]
    return years

# Expand year
expanded_rows = []
for idx, row in province_policy.iterrows():
    for y in expand_year_range(row):
        new_row = row.copy()
        new_row['active_year'] = y
        expanded_rows.append(new_row)

province_policy_panel = pd.DataFrame(expanded_rows)

# Drop 'year' 
province_policy_panel.drop(columns=['year'], inplace=True)

# Rename columns to match the demand policy DataFrame
province_policy_panel.rename(columns={'active_year': 'year'}, inplace=True)
# Extract provinces and years
province_list = df['province'].unique()
years = list(range(2019, 2024))

# Construct policy panel
demand_policy_df = pd.DataFrame([(province, year) for province in province_list for year in years],
                        columns=['province', 'year'])

# 将 year 转为整数并排序
demand_policy_df['year'] = demand_policy_df['year'].astype(int)
demand_policy_df = demand_policy_df.sort_values(['year', 'province']).reset_index(drop=True)

# 如果 df 也需要排序
df['year'] = df['year'].astype(int)
df = df.sort_values(['year', 'province']).reset_index(drop=True)

# Merge national policy data
demand_policy_df = demand_policy_df.merge(national_policy, on='year', how='left')

# Merge province policy data
demand_policy_df = demand_policy_df.merge(province_policy_panel, on=['province', 'year'], how='left')


In [34]:
# Fill all NaN values with 0
demand_policy_df.fillna(0, inplace=True)

# Convert subsidy to units of 10000 CNY
demand_policy_df['sub'] = demand_policy_df['sub'] / 10000

# Sort year of demand policy DataFrame
demand_policy_df['year'] = demand_policy_df['year'].astype(int)
demand_policy_df = demand_policy_df.sort_values(['year', 'province']).reset_index(drop=True)
df['year'] = df['year'].astype(int)
df = df.sort_values(['year', 'province']).reset_index(drop=True)

# Merge the demand policy DataFrame with the main DataFrame
df = df.merge(demand_policy_df, on=['province', 'year'], how='left')

In [35]:
################ Compute net prices of policies
# Define the function to compute net prices
def compute_net_price(share, manual_subsidy=None):
    rg = share['range']
    p = share['prices']
    # If manual_subsidy is provided, use its values; otherwise use from share
    if manual_subsidy is not None:
        sub = manual_subsidy.get('sub', share['sub'])
        nat = manual_subsidy.get('nat', share['nat'])
        PHEV = manual_subsidy.get('PHEV', share['sub_PHEV']) if share['fuel_type'] == 'PHEV' else 0
        BEV1 = manual_subsidy.get('BEV1', share['sub_BEV_1']) if share['fuel_type'] == 'BEV' else 0
        BEV2 = manual_subsidy.get('BEV2', share['sub_BEV_2']) if share['fuel_type'] == 'BEV' else 0
        BEV3 = manual_subsidy.get('BEV3', share['sub_BEV_3']) if share['fuel_type'] == 'BEV' else 0
        t = manual_subsidy.get('t', share['Tax'])
    else:
        sub = share['sub']
        nat = share['nat']
        PHEV = share['sub_PHEV'] if share['fuel_type'] == 'PHEV' else 0
        BEV1 = share['sub_BEV_1'] if share['fuel_type'] == 'BEV' else 0
        BEV2 = share['sub_BEV_2'] if share['fuel_type'] == 'BEV' else 0
        BEV3 = share['sub_BEV_3'] if share['fuel_type'] == 'BEV' else 0
        t = share['Tax']
    def range_category(r, BEV1, BEV2, BEV3):
        if 250 <= r < 300:
            return BEV1
        elif 300 <= r < 400:
            return BEV2
        elif 400 <= r:
            return BEV3
        else:
            return 0
    BEV = range_category(rg, BEV1, BEV2, BEV3)

    if nat == 0:
        return p * t - sub - PHEV - BEV
    elif nat == 1:
        return p * t - nat * 10000 * sub * (BEV + PHEV) - PHEV - BEV
    else:
        return 0


# Compute net prices
df['net_prices'] = df.apply(compute_net_price, axis=1)



In [36]:
################ Translation
# Translate all entries in columns with Chinese characters using the provided mapping
translations = {
    "type": {
        "国产新能源乘用车": "Domestic New Energy Passenger Vehicle",
        "国产燃油乘用车": "Domestic Fuel Passenger Vehicle"
    },
    "province": {
        "上海市": "Shanghai",
        "云南省": "Yunnan",
        "内蒙古自治区": "Inner Mongolia",
        "北京市": "Beijing",
        "吉林省": "Jilin",
        "四川省": "Sichuan",
        "天津市": "Tianjin",
        "宁夏回族自治区": "Ningxia",
        "安徽省": "Anhui",
        "山东省": "Shandong",
        "山西省": "Shanxi",
        "广东省": "Guangdong",
        "广西壮族自治区": "Guangxi",
        "新疆维吾尔自治区": "Xinjiang",
        "江苏省": "Jiangsu",
        "江西省": "Jiangxi",
        "河北省": "Hebei",
        "河南省": "Henan",
        "浙江省": "Zhejiang",
        "海南省": "Hainan",
        "湖北省": "Hubei",
        "湖南省": "Hunan",
        "甘肃省": "Gansu",
        "福建省": "Fujian",
        "西藏自治区": "Tibet",
        "贵州省": "Guizhou",
        "辽宁省": "Liaoning",
        "重庆市": "Chongqing",
        "陕西省": "Shaanxi",
        "青海省": "Qinghai",
        "黑龙江省": "Heilongjiang"
    },
    'model_translations' : {
    "景逸S50": "Jingyi S50",
    "云度π1": "Yudo π1",
    "云度π3": "Yudo π3",
    "嘉际": "Jiaji",
    "帝豪": "Emgrand",
    "帝豪GL": "Emgrand GL",
    "帝豪GSe": "Emgrand GSe",
    "星越": "Xingyue",
    "缤越": "Binyue",
    "奥迪A6L": "Audi A6L",
    "奥迪Q2L": "Audi Q2L",
    "宝马5系": "BMW 5 Series",
    "宝马X1": "BMW X1",
    "宝骏E100": "Baojun E100",
    "小鹏G3": "XPeng G3",
    "轩逸": "Sylphy",
    "欧拉iQ": "ORA iQ",
    "江淮iEV6E": "JAC iEV6E",
    "沃尔沃XC60": "Volvo XC60",
    "祺智EV": "GAC NE EV",
    "蒙迪欧": "Mondeo",
    "领界": "Territory",
    "红旗E-HS3": "Hongqi E-HS3",
    "腾势": "Denza",
    "荣威Ei5": "Roewe Ei5",
    "荣威RX5 eMAX": "Roewe RX5 eMAX",
    "蔚来ES6": "NIO ES6",
    "蔚来ES8": "NIO ES8",
    "起亚K3": "Kia K3",
    "起亚K5": "Kia K5",
    "逸动": "Eado",
    "长安CS75": "Changan CS75",
    "领克01": "Lynk & Co 01",
    "领克03": "Lynk & Co 03",
    "劲炫ASX": "ASX",
    "东南DX3": "Soueast DX3",
    "东南DX5": "Soueast DX5",
    "东南DX7": "Soueast DX7",
    "丰田C-HR": "Toyota C-HR",
    "五菱荣光": "Wuling Rongguang",
    "五菱荣光V": "Wuling Rongguang V",
    "传祺GS3": "Trumpchi GS3",
    "传祺GS5": "Trumpchi GS5",
    "凯翼X5": "Cowin X5",
    "凯迪拉克XT4": "Cadillac XT4",
    "凯迪拉克XT5": "Cadillac XT5",
    "凯迪拉克XT6": "Cadillac XT6",
    "昂科拉GX": "Encore GX",
    "远景X1": "Yuanjing X1",
    "远景X3": "Yuanjing X3",
    "启辰D60": "Venucia D60",
    "哈弗F7": "Haval F7",
    "哈弗H6": "Haval H6",
    "瑞虎3": "Tiggo 3",
    "瑞虎3X": "Tiggo 3X",
    "艾瑞泽GX": "Arrizo GX",
    "奔腾T33": "Bestune T33",
    "奥迪A3": "Audi A3",
    "奥迪Q3": "Audi Q3",
    "宝沃BX5": "Borgward BX5",
    "宝沃BX7": "Borgward BX7",
    "宝马3系": "BMW 3 Series",
    "宝马X2": "BMW X2",
    "宝马X3": "BMW X3",
    "宝骏310": "Baojun 310",
    "宝骏360": "Baojun 360",
    "宝骏530": "Baojun 530",
    "宝骏730": "Baojun 730",
    "宝骏RC-6": "Baojun RC-6",
    "宝骏RM-5": "Baojun RM-5",
    "宝骏RS-3": "Baojun RS-3",
    "东风小康C37": "Dongfeng Sokon C37",
    "风光580": "Fengguang 580",
    "风光S560": "Fengguang S560",
    "风光ix5": "Fengguang ix5",
    "捷豹XEL": "Jaguar XEL",
    "捷豹XFL": "Jaguar XFL",
    "捷途X70": "Jetour X70",
    "捷途X90": "Jetour X90",
    "星途TX": "Exeed TX",
    "本田CR-V": "Honda CR-V",
    "本田UR-V": "Honda UR-V",
    "本田XR-V": "Honda XR-V",
    "标致4008": "Peugeot 4008",
    "汉腾X5": "Hanteng X5",
    "汉腾X7": "Hanteng X7",
    "瑞风M3": "Refine M3",
    "瑞风S3": "Refine S3",
    "瑞风S7": "Refine S7",
    "锐际": "Escape",
    "伽途ix5": "Foton Gratour ix5",
    "红旗H5": "Hongqi H5",
    "红旗HS5": "Hongqi HS5",
    "红旗HS7": "Hongqi HS7",
    "荣威RX3": "Roewe RX3",
    "荣威RX5": "Roewe RX5",
    "荣威RX5 MAX": "Roewe RX5 MAX",
    "荣威RX8": "Roewe RX8",
    "荣威i5": "Roewe i5",
    "荣威i6": "Roewe i6",
    "观致3": "Qoros 3",
    "讴歌CDX": "Acura CDX",
    "讴歌RDX": "Acura RDX",
    "起亚KX5": "Kia KX5",
    "起亚KX7": "Kia KX7",
    "揽胜极光": "Range Rover Evoque",
    "斯派卡": "Space Star",
    "小海狮X30": "Xiaohaishi X30",
    "睿行S50": "Ruixing S50",
    "欧尚": "Oushang",
    "长行": "Changxing",
    "马自达CX-4": "Mazda CX-4",
    "马自达CX-5": "Mazda CX-5",
    "马自达CX-8": "Mazda CX-8",
    "江淮iEVA50": "JAC iEVA50",
    "易至EX5": "Yizhi EX5",
    "中华H530": "Zhonghua H530",
    "凯翼X3": "Cowin X3",
    "启腾EX80": "Qiteng EX80",
    "风光ix7": "Fengguang ix7",
    "风景V3": "Fengjing V3",
    "启辰E30": "Venucia E30",
    "思皓E20X": "Sehol E20X",
    "奇瑞eQ": "Chery eQ",
    "睿行ES30": "Ruixing ES30",
    "佳宝V80": "Jiabao V80",
    "五菱宏光": "Wuling Hongguang",
    "风光E1": "Fengguang E1",
    "合创007": "Hycan 007",
    "哪吒N01": "Neta N01",
    "哪吒U": "Neta U",
    "威马EX5": "WM Motor EX5",
    "宝骏E200": "Baojun E200",
    "小鹏P7": "XPeng P7",
    "枫叶30X": "Maple 30X",
    "欧拉好猫": "ORA Good Cat",
    "秦Pro": "Qin Pro",
    "沃尔沃XC40": "Volvo XC40",
    "爱驰U5": "Aiways U5",
    "理想ONE": "Li ONE",
    "北京X7": "Beijing X7",
    "腾势X": "Denza X",
    "赛力斯SF5": "Seres SF5",
    "零跑S01": "Leapmotor S01",
    "零跑T03": "Leapmotor T03",
    "领克06": "Lynk & Co 06",
    "风光330": "Fengguang 330",
    "奕炫": "Yixuan",
    "哈弗大狗": "Haval Big Dog",
    "探岳X": "Tanyue X",
    "途观X": "Tiguan X",
    "宝骏RC-5": "Baojun RC-5",
    "思皓X8": "Sehol X8",
    "捷达VA3": "Jetta VA3",
    "捷达VS5": "Jetta VS5",
    "捷达VS7": "Jetta VS7",
    "星途LX": "Exeed LX",
    "海马7X": "Haima 7X",
    "伊兰特": "Elantra",
    "红旗H9": "Hongqi H9",
    "荣威iMAX8": "Roewe iMAX8",
    "长安CS55PLUS": "Changan CS55 PLUS",
    "雪铁龙C3L": "Citroën C3L",
    "领克05": "Lynk & Co 05",
    "马自达CX-30": "Mazda CX-30",
    "风光E3": "Fengguang E3",
    "凌宝BOX": "Lingbao BOX",
    "易至EV3": "Yizhi EV3",
    "炫界": "Xuanjie",
    "野马EC60": "Yema EC60",
    "中华V5": "Zhonghua V5",
    "奔腾E01": "Bestune E01",
    "合创Z03": "Hycan Z03",
    "哪吒V": "Neta V",
    "大众ID.3": "Volkswagen ID.3",
    "威马W6": "WM Motor W6",
    "小鹏P5": "XPeng P5",
    "岚图FREE": "Voyah FREE",
    "思皓E40X": "Sehol E40X",
    "思皓E50A": "Sehol E50A",
    "极氪001": "Zeekr 001",
    "枫叶80V": "Maple 80V",
    "比亚迪D1": "BYD D1",
    "索纳塔": "Sonata",
    "红旗E-HS9": "Hongqi E-HS9",
    "红旗E-QM5": "Hongqi E-QM5",
    "发现运动版": "Discovery Sport",
    "零跑C11": "Leapmotor C11",
    "雷丁芒果": "Letin Mango",
    "坦克300": "Tank 300",
    "摩卡": "Mocha",
    "奕炫MAX": "Yixuan MAX",
    "五十铃MU-X": "Isuzu MU-X",
    "五菱征程": "Wuling Zhengcheng",
    "北京BJ30": "Beijing BJ30",
    "星越L": "Xingyue L",
    "远景X6": "Yuanjing X6",
    "启辰大V": "Venucia V",
    "哈弗神兽": "Haval Shenshou",
    "途昂X": "Teramont X",
    "艾瑞泽5 PLUS": "Arrizo 5 PLUS",
    "思皓QX": "Sehol QX",
    "长安CS75 PLUS": "Changan CS75 PLUS",
    "长安UNI-K": "Changan UNI-K",
    "迈锐宝XL": "Malibu XL",
    "凡尔赛C5 X": "C5 X",
    "领克09": "Lynk & Co 09",
    "QQ冰淇淋": "QQ Ice Cream",
    "思皓X4": "Sehol X4",
    "思皓X7": "Sehol X7",
    "小虎FEV": "Xiaohu FEV",
    "北京EU5": "Beijing EU5",
    "科莱威CLEVER": "Kaleiwei CLEVER",
    "北汽新能源EC3": "BAIC BJEV EC3",
    "海马6P": "Haima 6P",
    "smart精灵#1": "smart #1",
    "一汽丰田bZ4X": "FAW Toyota bZ4X",
    "广汽丰田bZ4X": "GAC Toyota bZ4X",
    "LYRIQ锐歌": "LYRIQ",
    "哪吒S": "Neta S",
    "天际ME5": "Skyworth ME5",
    "奔驰EQE": "Mercedes-Benz EQE",
    "奥迪Q4 e-tron": "Audi Q4 e-tron",
    "奥迪Q5 e-tron": "Audi Q5 e-tron",
    "奥迪e-tron": "Audi e-tron",
    "宝马i3": "BMW i3",
    "富康ES600": "Fukang ES600",
    "小鹏G9": "XPeng G9",
    "智己L7": "IM L7",
    "朋克多多": "Punk Duoduo",
    "朋克美美": "Punk Meimei",
    "本田e:NS1": "Honda e:NS1",
    "枫叶60s": "Maple 60s",
    "欧拉芭蕾猫": "ORA Ballet Cat",
    "欧拉闪电猫": "ORA Lightning Cat",
    "护卫舰07": "Frigate 07",
    "驱逐舰05": "Destroyer 05",
    "沃尔沃C40": "Volvo C40",
    "爱驰U6": "Aiways U6",
    "理想L8": "Li L8",
    "理想L9": "Li L9",
    "YOUNG光小新": "YOUNG Guangxiaoxin",
    "腾势D9": "Denza D9",
    "问界M5": "AITO M5",
    "问界M7": "AITO M7",
    "阿维塔11": "Avatr 11",
    "零跑C01": "Leapmotor C01",
    "飞凡R7": "Rising Auto R7",
    "东南DX8": "Soueast DX8",
    "风行T5 EVO": "Forthing T5 EVO",
    "五菱宏光V": "Wuling Hongguang V",
    "帝豪S": "Emgrand S",
    "名爵HS": "MG HS",
    "哈弗酷狗": "Haval Cool Dog",
    "坦克500": "Tank 500",
    "宝马X5": "BMW X5",
    "思皓X6": "Sehol X6",
    "红旗HQ9": "Hongqi HQ9",
    "红旗LS7": "Hongqi LS7",
    "长安CS35PLUS": "Changan CS35 PLUS",
    "元宝": "Yuanbao",
    "骏行": "Junxing",
    "远志M1": "Yuanzhi M1",
    "富康ES500": "Fukang ES500",
    "瑞风E3": "Refine E3",
    "百智大熊": "Baizhi Big Bear",
    "创维EV6": "Skyworth EV6",
    "风行雷霆": "Forthing Thunder",
    "五菱缤果": "Wuling Bingo",
    "五菱荣光EV": "Wuling Rongguang EV",
    "传祺E9": "Trumpchi E9",
    "传祺ES9": "Trumpchi ES9",
    "创维HT-i": "Skyworth HT-i",
    "合创A06": "Hycan A06",
    "银河L6": "Galaxy L6",
    "银河L7": "Galaxy L7",
    "哈弗枭龙MAX": "Haval Xiaolong MAX",
    "哪吒AYA": "Neta AYA",
    "Aion LX": "Aion LX",
    "ID.4 CROZZ": "ID.4 CROZZ",
    "ID.4 X": "ID.4 X",
    "ID.6 CROZZ": "ID.6 CROZZ",
    "ID.6 X": "ID.6 X",
    "奔驰EQA": "Mercedes-Benz EQA",
    "奔驰EQB": "Mercedes-Benz EQB",
    "奔驰EQC": "Mercedes-Benz EQC",
    "宝骏云朵": "Baojun Cloud",
    "小鹏G6": "XPeng G6",
    "Ariya艾睿雅": "Ariya",
    "智己LS6": "IM LS6",
    "智己LS7": "IM LS7",
    "极氪009": "Zeekr 009",
    "极氪X": "Zeekr X",
    "元PLUS": "Yuan PLUS",
    "汉": "Han",
    "海豚": "Dolphin",
    "海豹": "Seal",
    "秦PLUS": "Qin PLUS",
    "钇为3": "Yiwei 3",
    "深蓝S7": "Deepal S7",
    "Model Y": "Model Y",
    "理想L7": "Li L7",
    "蓝电E5": "Landian E5",
    "深蓝SL03": "Deepal SL03",
    "飞凡F7": "Rising Auto F7",
    "高合HiPhi Y": "HiPhi Y",
    "风光380": "Fengguang 380",
    "北京X3": "Beijing X3",
    "探索06": "Tansuo 06",
    "LIFE": "Life",
    "本田HR-V": "Honda HR-V",
    "标致408X": "Peugeot 408X",
    "江淮A5 PLUS": "JAC A5 PLUS",
    "红旗HS3": "Hongqi HS3",
    "荣威RX9": "Roewe RX9",
    "炫界Pro EV": "Xuanjie Pro EV",
    "家宝": "Jiabao",
    "启辰T60EV": "Venucia T60EV",
    "悦虎": "Yuehu",
    "小蚂蚁": "Little Ant",
    "羿": "Yi",
    "风行S50EV": "Forthing S50EV",
    "e爱丽舍": "e-Elysee",
    "岚图追光": "Voyah Zhuiguang",
    "捷途X70S EV": "Jetour X70S EV"
}
    }

for col in translations:
    # If the column exists in df, map it
    if col in df.columns:
        df[col] = df[col].map(translations[col]).fillna(df[col])
    # Special case for 'model_translations': map 'model' column
    elif col == 'model_translations' and 'model' in df.columns:
        df['model'] = df['model'].map(translations['model_translations']).fillna(df['model'])


In [37]:
# Export df as csv
df.to_csv('model_ready.csv', index=False)

In [38]:
################ Prepare supply-side data with charging policy
supply_df = pd.read_csv('supply_side_data.csv')
charging_policy_df = pd.read_excel(r"C:\Users\Lenovo\Desktop\Dissertaion\China Data\Charging\Charging_policy.xlsx")

def expand_year_range(row):
    years = []
    if pd.notnull(row['year']):
        parts = str(row['year']).split('-')
        if len(parts) == 2:
            start, end = int(parts[0]), int(parts[1])
            years = list(range(start, end + 1))
        else:
            years = [int(parts[0])]
    return years

# 展开年份区间
expanded_rows = []
for idx, row in charging_policy_df.iterrows():
    for y in expand_year_range(row):
        new_row = row.copy()
        new_row['active_year'] = y
        expanded_rows.append(new_row)

charging_policy_df = pd.DataFrame(expanded_rows)

charging_policy_df = charging_policy_df[['province', 'active_year', 'sub_fix', 'sub_ope']]

# Rename active_year to year for consistency
charging_policy_df.rename(columns={'active_year': 'year'}, inplace=True)

# Merge charging policy data with supply-side data
supply_df = supply_df.merge(charging_policy_df, on=['province', 'year'], how='left')
supply_df['sub_fix'] = supply_df['sub_fix'].fillna(0)
supply_df['sub_ope'] = supply_df['sub_ope'].fillna(0)

# Translate all entries in columns with Chinese characters in supply_df using the same mapping

for col in translations:
    if col in supply_df.columns:
        supply_df[col] = supply_df[col].map(translations[col]).fillna(supply_df[col])

In [39]:
# Translate all entries in columns with Chinese characters in supply_df using the same mapping

for col in translations:
    if col in supply_df.columns:
        supply_df[col] = supply_df[col].map(translations[col]).fillna(supply_df[col])
    # Special case for 'model_translations': map 'model' column
    elif col == 'model_translations' and 'model' in supply_df.columns:
        supply_df['model'] = supply_df['model'].map(translations['model_translations']).fillna(supply_df['model'])

In [40]:
# Export supply_df as csv
supply_df.to_csv('supply_ready.csv', index=False)